# Task B Notebook v0 — Fast Recommendation Iteration

Goal:
1. Reuse the filtered Task A iteration dataset.
2. Build quick candidate generators for normal, cold-start, and cross-domain routes.
3. Evaluate Hit@10 / NDCG@10 / Recall@30 on small samples.
4. Optionally rerank top 30 candidates with Gemini or DeepSeek.

This notebook is intentionally compact and iteration-friendly. Once the method is stable, we can convert the best pieces into clean scripts for the API/container.

In [1]:
# Optional, only run if needed:
# %pip install pandas pyarrow scikit-learn scipy requests tqdm

import os
import json
import math
import time
import re
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm

from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 180)

In [2]:
# -------------------------
# Config
# -------------------------

TASK_DATA_DIR = Path("data/processed/task_a_iteration_v0")
CACHE_DIR = Path("data/processed/baseline_v0/cache")
TASK_B_CACHE_DIR = CACHE_DIR / "task_b"
TASK_B_CACHE_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = TASK_DATA_DIR / "train.parquet"
VAL_PATH = TASK_DATA_DIR / "internal_val.parquet"
SHADOW_PATH = TASK_DATA_DIR / "shadow_test.parquet"
ITEMS_PATH = TASK_DATA_DIR / "items.parquet"

TOP_CANDIDATES = 30
TOP_K = 10
RANDOM_SEED = 42

# Candidate source sizes. Increase after smoke tests.
N_CF = 120
N_SEMANTIC = 120
N_POPULAR = 80
N_GATEWAY = 80

# Switch as needed.
LLM_PROVIDER = os.getenv("LLM_PROVIDER", "gemini")  # "gemini" or "deepseek"
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-2.5-flash-lite")
DEEPSEEK_MODEL = os.getenv("DEEPSEEK_MODEL", "deepseek-v4-flash")
TEMPERATURE = 0.1
MAX_OUTPUT_TOKENS = 2200
LLM_RETRIES = 2

print("Train exists:", TRAIN_PATH.exists())
print("Val exists:", VAL_PATH.exists())
print("Items exists:", ITEMS_PATH.exists())
print("Provider:", LLM_PROVIDER)

Train exists: True
Val exists: True
Items exists: True
Provider: gemini


In [3]:
# -------------------------
# Load filtered data from Task A iteration
# -------------------------

train_df = pd.read_parquet(TRAIN_PATH)
val_df = pd.read_parquet(VAL_PATH)
shadow_df = pd.read_parquet(SHADOW_PATH) if SHADOW_PATH.exists() else None
items_df = pd.read_parquet(ITEMS_PATH)

# Standardize keys as strings.
for df in [train_df, val_df, items_df] + ([] if shadow_df is None else [shadow_df]):
    df["domain"] = df["domain"].astype(str)
    df["parent_asin"] = df["parent_asin"].astype(str)
    if "user_id" in df.columns:
        df["user_id"] = df["user_id"].astype(str)

items_df = items_df.drop_duplicates(["domain", "parent_asin"]).copy()

print("train:", train_df.shape)
print("val:", val_df.shape)
print("shadow:", None if shadow_df is None else shadow_df.shape)
print("items:", items_df.shape)
print("train users:", train_df["user_id"].nunique())
print("val users:", val_df["user_id"].nunique())
print("domains:", train_df["domain"].value_counts().to_dict())

items_df.head(2)

train: (190457, 13)
val: (20408, 13)
shadow: (17348, 13)
items: (50927, 11)
train users: 5978
val users: 5978
domains: {'Movies_and_TV': 104696, 'Books': 85761}


,domain,parent_asin,main_category,title,store,categories,features,description,price,has_useful_metadata,n_train_reviews_for_item
0,Books,0316185361,Books,Service: A Navy SEAL at War,"Marcus Luttrell (Author), James D. Hornfischer",Books Biographies & Memoirs Leaders & Notable People,"Marcus Luttrell, author of the #1 bestseller Lone Survivor , share war stories about true American heroism from himself and other soldiers who bravely fought alongside him. Nav...","Review Praise for SERVICE""An action-packed...reflective saga of contemporary military service.""― Kirkus Reviews ""Marcus Luttrell, with James D. Hornfischer, has written another...",17.17,True,4
1,Books,1932225323,Books,Four Centuries of American Education,David Barton (Author),Books Education & Teaching Schools & Teaching,"For four centuries, religion, morality, and knowledge formed the core elements of American education, but in recent decades, a secularized approach to education has gained prom...","About the Author David Barton is the founder of WallBuilders, an organization dedicated to presenting America's forgotten history and heroes, with an emphasis on our moral, rel...",6.99,True,3


In [4]:
# -------------------------
# Small helpers
# -------------------------

def clean_text(x: Any, max_chars: Optional[int] = None) -> str:
    if x is None or (isinstance(x, float) and math.isnan(x)):
        s = ""
    elif isinstance(x, list):
        s = " ".join(clean_text(v) for v in x)
    elif isinstance(x, dict):
        s = json.dumps(x, ensure_ascii=False)
    else:
        s = str(x)
    s = re.sub(r"\s+", " ", s).strip()
    if max_chars and len(s) > max_chars:
        return s[:max_chars].rsplit(" ", 1)[0] + "..."
    return s


def safe_json_loads(text: str) -> Dict[str, Any]:
    if text is None:
        raise ValueError("LLM returned empty response")
    text = str(text).strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE).strip()
    text = re.sub(r"\s*```$", "", text).strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        start = text.find("{")
        end = text.rfind("}")
        if start != -1 and end != -1 and end > start:
            return json.loads(text[start:end + 1])
        raise


def make_item_id(domain: str, parent_asin: str) -> str:
    return f"{domain}||{parent_asin}"


def split_item_id(item_id: str) -> Tuple[str, str]:
    domain, parent_asin = item_id.split("||", 1)
    return domain, parent_asin


def item_text_from_row(row: pd.Series, max_chars: int = 1600) -> str:
    parts = []
    for col in ["title", "main_category", "categories", "features", "description", "store"]:
        if col in row:
            parts.append(clean_text(row.get(col, ""), max_chars=500))
    return clean_text(" | ".join([p for p in parts if p]), max_chars=max_chars)


def hit_ndcg(candidate_ids: List[str], target_id: str, k: int = 10) -> Dict[str, float]:
    top = candidate_ids[:k]
    if target_id not in top:
        return {"hit": 0.0, "ndcg": 0.0}
    rank = top.index(target_id) + 1
    return {"hit": 1.0, "ndcg": 1.0 / math.log2(rank + 1)}


def recall_at_k(candidate_ids: List[str], target_id: str, k: int = 30) -> float:
    return float(target_id in candidate_ids[:k])

In [5]:
# -------------------------
# Item table + item stats
# -------------------------

items_df = items_df.copy()
items_df["item_id"] = items_df.apply(lambda r: make_item_id(r["domain"], r["parent_asin"]), axis=1)
items_df["item_text"] = items_df.apply(item_text_from_row, axis=1)

item_lookup = {
    r.item_id: r._asdict()
    for r in items_df.fillna("").itertuples(index=False)
}

train_tmp = train_df.copy()
train_tmp["item_id"] = train_tmp.apply(lambda r: make_item_id(r["domain"], r["parent_asin"]), axis=1)

# Basic popularity and quality from train only.
item_stats = (
    train_tmp.groupby(["domain", "parent_asin", "item_id"])
    .agg(
        n_train=("rating", "size"),
        mean_rating=("rating", "mean"),
        sum_rating=("rating", "sum"),
    )
    .reset_index()
)

global_mean = float(train_tmp["rating"].mean())
REG_M = 10.0
item_stats["regularized_rating"] = (item_stats["sum_rating"] + REG_M * global_mean) / (item_stats["n_train"] + REG_M)
item_stats["popularity_score"] = item_stats["regularized_rating"] * np.log1p(item_stats["n_train"])

# Gateway score: items liked by users in their first few known reviews.
early = (
    train_tmp.sort_values(["user_id", "timestamp", "domain", "parent_asin"])
    .groupby("user_id")
    .head(3)
)

gateway = (
    early.groupby(["domain", "parent_asin", "item_id"])
    .agg(n_gateway=("rating", "size"), mean_gateway_rating=("rating", "mean"))
    .reset_index()
)
gateway["gateway_score"] = gateway["mean_gateway_rating"] * np.log1p(gateway["n_gateway"])

item_stats = item_stats.merge(
    gateway[["item_id", "n_gateway", "mean_gateway_rating", "gateway_score"]],
    on="item_id",
    how="left",
)
for col in ["n_gateway", "mean_gateway_rating", "gateway_score"]:
    item_stats[col] = item_stats[col].fillna(0.0)

item_stats = item_stats.merge(
    items_df[["item_id", "title", "categories", "item_text"]],
    on="item_id",
    how="left",
)

print("item_stats:", item_stats.shape)
item_stats.sort_values("popularity_score", ascending=False).head(5)

item_stats: (47639, 14)


,domain,parent_asin,item_id,n_train,mean_rating,sum_rating,regularized_rating,popularity_score,n_gateway,mean_gateway_rating,gateway_score,title,categories,item_text
20617,Books,1780671067,Books||1780671067,184,4.706522,866.0,4.691284,24.490169,26.0,4.615385,15.211555,Secret Garden: An Inky Treasure Hunt and Coloring Book for Adults,Books Arts & Photography Graphic Design,Secret Garden: An Inky Treasure Hunt and Coloring Book for Adults | Books | Books Arts & Photography Graphic Design | Secret Garden: An Inky Treasure Hunt and Colouring Book by...
9482,Books,0679805273,Books||0679805273,130,4.861538,632.0,4.829350,23.544035,21.0,4.904762,15.160827,"Oh, the Places You'll Go!",Books Children's Books Classics,"Oh, the Places You'll Go! | Books | Books Children's Books Classics | Dr. Seuss’s wonderfully wise Oh, the Places You’ll Go! is the perfect gift to celebrate all of our special..."
15797,Books,1442450703,Books||1442450703,107,4.878505,522.0,4.838539,22.654672,8.0,5.000000,10.986123,Chicka Chicka Boom Boom (Board Book),Books Children's Books Literature & Fiction,Chicka Chicka Boom Boom (Board Book) | Books | Books Children's Books Literature & Fiction | There is always enough room on your child’s bookshelf for this Classic Board Book e...
44426,Movies_and_TV,B00YHRMMLC,Movies_and_TV||B00YHRMMLC,146,4.513699,659.0,4.507109,22.492424,11.0,4.636364,11.520931,San Andreas (2 Disc/DVD),Movies & TV Fully Loaded DVDs Special Editions,"San Andreas (2 Disc/DVD) | Movies & TV | Movies & TV Fully Loaded DVDs Special Editions | San Andreas (DVD) ]]> | Brad Peyton (Director), Dwayne ""The Rock"" Johnson (Actor), Car..."
20618,Books,1780674880,Books||1780674880,92,4.804348,442.0,4.765775,21.601348,15.0,4.933333,13.678104,"Enchanted Forest: An Inky Quest and Coloring book (Activity Books, Mindfulness and Meditation, Illustrated Floral Prints)","Books Children's Books Activities, Crafts & Games","Enchanted Forest: An Inky Quest and Coloring book (Activity Books, Mindfulness and Meditation, Illustrated Floral Prints) | Books | Books Children's Books Activities, Crafts & ..."


In [7]:
# -------------------------
# Build TF-IDF semantic model over item metadata
# -------------------------

vectorizer = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    min_df=2,
    stop_words="english",
)

item_texts = items_df["item_text"].fillna("").astype(str).tolist()
item_tfidf = vectorizer.fit_transform(item_texts)

item_ids = items_df["item_id"].tolist()
item_id_to_idx = {item_id: i for i, item_id in enumerate(item_ids)}
idx_to_item_id = {i: item_id for item_id, i in item_id_to_idx.items()}

print("TF-IDF matrix:", item_tfidf.shape)

TF-IDF matrix: (50927, 50000)


In [8]:
# -------------------------
# Build quick collaborative filtering model with TruncatedSVD
# -------------------------

user_ids = sorted(train_tmp["user_id"].unique().tolist())
cf_item_ids = item_ids  # align CF item columns with items_df / TF-IDF IDs

user_to_idx = {u: i for i, u in enumerate(user_ids)}
cf_item_to_idx = {it: i for i, it in enumerate(cf_item_ids)}
idx_to_cf_item = {i: it for it, i in cf_item_to_idx.items()}

cf_rows = train_tmp[train_tmp["item_id"].isin(cf_item_to_idx)].copy()
row_idx = cf_rows["user_id"].map(user_to_idx).to_numpy()
col_idx = cf_rows["item_id"].map(cf_item_to_idx).to_numpy()
values = cf_rows["rating"].astype(float).to_numpy()

R = sparse.csr_matrix((values, (row_idx, col_idx)), shape=(len(user_ids), len(cf_item_ids)))

N_COMPONENTS = 64
svd = TruncatedSVD(n_components=N_COMPONENTS, random_state=RANDOM_SEED)
user_latent = svd.fit_transform(R)      # shape: users x components
item_latent = svd.components_.T         # shape: items x components

print("R:", R.shape, "nnz:", R.nnz)
print("user_latent:", user_latent.shape, "item_latent:", item_latent.shape)

R: (5978, 50927) nnz: 189676
user_latent: (5978, 64) item_latent: (50927, 64)


In [9]:
# -------------------------
# Fast lookup structures
# -------------------------

user_seen_items = train_tmp.groupby("user_id")["item_id"].apply(set).to_dict()
user_train_domains = train_tmp.groupby("user_id")["domain"].apply(lambda s: sorted(set(s))).to_dict()

# Per-user history dataframes are handy during iteration.
train_by_user = {u: g.sort_values(["timestamp", "domain", "parent_asin"]).copy() for u, g in train_tmp.groupby("user_id")}

# Domain item ID lists for filtering.
domain_to_item_ids = items_df.groupby("domain")["item_id"].apply(list).to_dict()

print("users with train history:", len(train_by_user))
print("domain item counts:", {k: len(v) for k, v in domain_to_item_ids.items()})

users with train history: 5978
domain item counts: {'Books': 28754, 'Movies_and_TV': 22173}


In [10]:
# -------------------------
# Candidate utility functions
# -------------------------

def _domain_filter_mask(domain: Optional[str]) -> Optional[np.ndarray]:
    if domain is None:
        return None
    return np.array([item_lookup[it]["domain"] == domain for it in item_ids])


def _exclude_seen_and_take(scores: np.ndarray, user_id: Optional[str], top_n: int, domain: Optional[str] = None) -> List[Tuple[str, float]]:
    scores = np.asarray(scores, dtype=float).copy()

    # Domain filter.
    if domain is not None:
        mask = _domain_filter_mask(domain)
        scores[~mask] = -np.inf

    # Remove seen train items.
    if user_id is not None:
        for it in user_seen_items.get(str(user_id), set()):
            idx = item_id_to_idx.get(it)
            if idx is not None:
                scores[idx] = -np.inf

    if not np.isfinite(scores).any():
        return []

    n = min(top_n, np.isfinite(scores).sum())
    candidate_idx = np.argpartition(-scores, n - 1)[:n]
    candidate_idx = candidate_idx[np.argsort(-scores[candidate_idx])]
    return [(idx_to_item_id[int(i)], float(scores[int(i)])) for i in candidate_idx if np.isfinite(scores[int(i)])]


def _candidates_df(rows: List[Tuple[str, float]], score_col: str, source: str) -> pd.DataFrame:
    out = []
    for item_id, score in rows:
        domain, parent_asin = split_item_id(item_id)
        meta = item_lookup.get(item_id, {})
        out.append({
            "item_id": item_id,
            "domain": domain,
            "parent_asin": parent_asin,
            "title": clean_text(meta.get("title", ""), 180),
            "categories": clean_text(meta.get("categories", ""), 260),
            score_col: score,
            "sources": source,
        })
    return pd.DataFrame(out)


def _minmax(s: pd.Series) -> pd.Series:
    s = pd.to_numeric(s, errors="coerce").fillna(0.0)
    lo, hi = float(s.min()), float(s.max())
    if hi <= lo:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - lo) / (hi - lo)


def blend_candidate_frames(frames: List[pd.DataFrame], top_n: int = TOP_CANDIDATES) -> pd.DataFrame:
    frames = [f for f in frames if f is not None and len(f) > 0]
    if not frames:
        return pd.DataFrame()

    combined = pd.concat(frames, ignore_index=True)

    # Merge duplicate item candidates and preserve source labels.
    score_cols = [c for c in ["cf_score", "semantic_score", "popularity_score", "gateway_score"] if c in combined.columns]
    agg = {c: "max" for c in score_cols}
    agg.update({"domain": "first", "parent_asin": "first", "title": "first", "categories": "first"})
    grouped = combined.groupby("item_id", as_index=False).agg(agg)

    source_map = combined.groupby("item_id")["sources"].apply(lambda s: ",".join(sorted(set(",".join(s).split(","))))).to_dict()
    grouped["sources"] = grouped["item_id"].map(source_map)

    for c in ["cf_score", "semantic_score", "popularity_score", "gateway_score"]:
        if c not in grouped.columns:
            grouped[c] = 0.0
        grouped[c] = pd.to_numeric(grouped[c], errors="coerce").fillna(0.0)
        grouped[c + "_norm"] = _minmax(grouped[c])

    grouped["final_score"] = (
        0.42 * grouped["cf_score_norm"] +
        0.33 * grouped["semantic_score_norm"] +
        0.18 * grouped["popularity_score_norm"] +
        0.07 * grouped["gateway_score_norm"]
    )

    return grouped.sort_values("final_score", ascending=False).head(top_n).reset_index(drop=True)

In [11]:
# -------------------------
# Candidate generators
# -------------------------

def cf_candidates(user_id: str, target_domain: Optional[str] = None, n: int = N_CF) -> pd.DataFrame:
    user_id = str(user_id)
    if user_id not in user_to_idx:
        return pd.DataFrame()
    uidx = user_to_idx[user_id]
    scores = user_latent[uidx] @ item_latent.T
    rows = _exclude_seen_and_take(scores, user_id=user_id, top_n=n, domain=target_domain)
    return _candidates_df(rows, "cf_score", "cf")


def semantic_candidates_from_text(query_text: str, user_id: Optional[str] = None, target_domain: Optional[str] = None, n: int = N_SEMANTIC) -> pd.DataFrame:
    query_text = clean_text(query_text, max_chars=4000)
    if not query_text:
        return pd.DataFrame()
    q = vectorizer.transform([query_text])
    scores = (item_tfidf @ q.T).toarray().ravel()
    rows = _exclude_seen_and_take(scores, user_id=user_id, top_n=n, domain=target_domain)
    return _candidates_df(rows, "semantic_score", "semantic")


def semantic_candidates_for_user(user_id: str, target_domain: Optional[str] = None, source_domain: Optional[str] = None, n: int = N_SEMANTIC) -> pd.DataFrame:
    hist = train_by_user.get(str(user_id))
    if hist is None or hist.empty:
        return pd.DataFrame()
    if source_domain is not None:
        hist = hist[hist["domain"] == source_domain]
    if hist.empty:
        return pd.DataFrame()

    # Use liked items as the user semantic query. Fall back to all history if needed.
    liked = hist[hist["rating"].astype(float) >= 4.0]
    if liked.empty:
        liked = hist

    idxs = [item_id_to_idx[it] for it in liked["item_id"].tolist() if it in item_id_to_idx]
    if not idxs:
        return pd.DataFrame()

    weights = liked[liked["item_id"].isin(item_id_to_idx)].copy()
    weights = weights["rating"].astype(float).to_numpy()
    weights = np.maximum(weights - 3.0, 0.2)
    weights = weights[:len(idxs)]

    q = item_tfidf[idxs].multiply(weights.reshape(-1, 1)).sum(axis=0)
    q = sparse.csr_matrix(q)
    scores = (item_tfidf @ q.T).toarray().ravel()
    rows = _exclude_seen_and_take(scores, user_id=str(user_id), top_n=n, domain=target_domain)
    return _candidates_df(rows, "semantic_score", "semantic_user")


def popular_candidates(target_domain: Optional[str] = None, user_id: Optional[str] = None, n: int = N_POPULAR) -> pd.DataFrame:
    df = item_stats.copy()
    if target_domain is not None:
        df = df[df["domain"] == target_domain]
    if user_id is not None:
        seen = user_seen_items.get(str(user_id), set())
        df = df[~df["item_id"].isin(seen)]
    df = df.sort_values("popularity_score", ascending=False).head(n)
    return _candidates_df(list(zip(df["item_id"], df["popularity_score"])), "popularity_score", "popular")


def gateway_candidates(target_domain: Optional[str] = None, user_id: Optional[str] = None, n: int = N_GATEWAY) -> pd.DataFrame:
    df = item_stats.copy()
    if target_domain is not None:
        df = df[df["domain"] == target_domain]
    if user_id is not None:
        seen = user_seen_items.get(str(user_id), set())
        df = df[~df["item_id"].isin(seen)]
    df = df.sort_values("gateway_score", ascending=False).head(n)
    return _candidates_df(list(zip(df["item_id"], df["gateway_score"])), "gateway_score", "gateway")


def normal_candidates(user_id: str, target_domain: Optional[str] = None, top_n: int = TOP_CANDIDATES) -> pd.DataFrame:
    frames = [
        cf_candidates(user_id, target_domain=target_domain),
        semantic_candidates_for_user(user_id, target_domain=target_domain),
        popular_candidates(target_domain=target_domain, user_id=user_id),
        gateway_candidates(target_domain=target_domain, user_id=user_id, n=40),
    ]
    out = blend_candidate_frames(frames, top_n=top_n)
    out["route"] = "normal"
    out["user_id"] = str(user_id)
    return out


def make_simulated_onboarding_text(user_id: str, max_items: int = 8) -> str:
    """Simulate lightweight onboarding from known history for offline testing only."""
    hist = train_by_user.get(str(user_id))
    if hist is None or hist.empty:
        return ""
    liked = hist[hist["rating"].astype(float) >= 4.0].sort_values("rating", ascending=False).head(max_items)
    bits = []
    for _, row in liked.iterrows():
        meta = item_lookup.get(row["item_id"], {})
        bits.append(f"Liked {row['domain']} item: {clean_text(meta.get('title',''), 120)} | {clean_text(meta.get('categories',''), 160)}")
    return "The user says they tend to like: " + " ; ".join(bits)


def cold_start_candidates(
    target_domain: Optional[str] = None,
    user_id: Optional[str] = None,
    onboarding_text: Optional[str] = None,
    use_onboarding: bool = True,
    use_gateway: bool = True,
    top_n: int = TOP_CANDIDATES,
) -> pd.DataFrame:
    frames = [popular_candidates(target_domain=target_domain, user_id=None)]
    if use_gateway:
        frames.append(gateway_candidates(target_domain=target_domain, user_id=None))
    if use_onboarding and onboarding_text:
        frames.append(semantic_candidates_from_text(onboarding_text, user_id=None, target_domain=target_domain))

    out = blend_candidate_frames(frames, top_n=top_n)
    out["route"] = "cold_start"
    out["user_id"] = "" if user_id is None else str(user_id)
    return out


def make_domain_agnostic_query(user_id: str, source_domain: str, max_items: int = 10) -> str:
    hist = train_by_user.get(str(user_id))
    if hist is None or hist.empty:
        return ""
    hist = hist[hist["domain"] == source_domain]
    liked = hist[hist["rating"].astype(float) >= 4.0].sort_values(["rating", "timestamp"], ascending=[False, False]).head(max_items)
    if liked.empty:
        liked = hist.tail(max_items)
    bits = []
    for _, row in liked.iterrows():
        meta = item_lookup.get(row["item_id"], {})
        bits.append(f"{clean_text(meta.get('title',''), 140)} | {clean_text(meta.get('categories',''), 220)} | review: {clean_text(row.get('review_text',''), 220)}")
    return f"Domain-agnostic transferable preferences inferred from {source_domain}: " + " ; ".join(bits)


def cross_domain_candidates(user_id: str, source_domain: str, target_domain: str, top_n: int = TOP_CANDIDATES) -> pd.DataFrame:
    query = make_domain_agnostic_query(user_id, source_domain=source_domain)
    frames = [
        semantic_candidates_from_text(query, user_id=user_id, target_domain=target_domain),
        popular_candidates(target_domain=target_domain, user_id=user_id),
        gateway_candidates(target_domain=target_domain, user_id=user_id),
    ]
    out = blend_candidate_frames(frames, top_n=top_n)
    out["route"] = "cross_domain"
    out["user_id"] = str(user_id)
    out["source_domain"] = source_domain
    out["target_domain"] = target_domain
    return out

In [16]:
# -------------------------
# Smoke test candidate generation on one validation row
# -------------------------

sample_row = val_df.sample(1, random_state=8).iloc[0]
uid = sample_row["user_id"]
target_domain = sample_row["domain"]
target_id = make_item_id(sample_row["domain"], sample_row["parent_asin"])

print("user:", uid)
print("target:", target_id, "rating:", sample_row["rating"])
print("train domains:", user_train_domains.get(uid))

cands = normal_candidates(uid, target_domain=target_domain, top_n=TOP_CANDIDATES)
print("target in top30:", target_id in cands["item_id"].tolist())
cands.head(10)

user: AFZTV7WTRNPPLF2BSJ4C2VBQRMOQ
target: Books||0142410381 rating: 5.0
train domains: ['Books', 'Movies_and_TV']
target in top30: False


,item_id,cf_score,semantic_score,popularity_score,gateway_score,domain,parent_asin,title,categories,sources,cf_score_norm,semantic_score_norm,popularity_score_norm,gateway_score_norm,final_score,route,user_id
0,Books||0399226907,0.818209,0.000000,20.930775,0.000000,Books,0399226907,The Very Hungry Caterpillar,Books Children's Books Early Learning,"cf,popular",1.000000,0.000000,0.854660,0.000000,0.573839,normal,AFZTV7WTRNPPLF2BSJ4C2VBQRMOQ
1,Books||0805047905,0.422971,0.982902,18.879907,0.000000,Books,0805047905,"Brown Bear, Brown Bear, What Do You See?",Books Children's Books Early Learning,"cf,popular,semantic_user",0.516947,0.407607,0.770918,0.000000,0.490393,normal,AFZTV7WTRNPPLF2BSJ4C2VBQRMOQ
2,Books||0679805273,0.466364,0.000000,23.544035,15.160827,Books,0679805273,"Oh, the Places You'll Go!",Books Children's Books Classics,"cf,gateway,popular",0.569981,0.000000,0.961367,0.996665,0.482205,normal,AFZTV7WTRNPPLF2BSJ4C2VBQRMOQ
3,Books||068983568X,0.000000,2.388536,15.680185,0.000000,Books,068983568X,Chicka Chicka Boom Boom,Books Children's Books Literature & Fiction,"popular,semantic_user",0.000000,0.990519,0.640264,0.000000,0.442119,normal,AFZTV7WTRNPPLF2BSJ4C2VBQRMOQ
4,Books||0545392551,0.541272,0.000000,20.742124,0.000000,Books,0545392551,Giraffes Can't Dance (Board Book),"Books Children's Books Arts, Music & Photography","cf,popular",0.661533,0.000000,0.846957,0.000000,0.430296,normal,AFZTV7WTRNPPLF2BSJ4C2VBQRMOQ
5,Books||0803736800,0.326667,0.000000,21.453394,11.829712,Books,0803736800,Dragons Love Tacos,"Books Children's Books Fairy Tales, Folk Tales & Myths","cf,gateway,popular",0.399247,0.000000,0.876000,0.777679,0.379801,normal,AFZTV7WTRNPPLF2BSJ4C2VBQRMOQ
6,Books||0671449028,0.328660,0.000000,17.688951,9.729551,Books,0671449028,The Going To Bed Book,Books Children's Books Early Learning,"cf,gateway,popular",0.401682,0.000000,0.722288,0.639616,0.343491,normal,AFZTV7WTRNPPLF2BSJ4C2VBQRMOQ
7,Books||0920668372,0.320921,0.000000,18.037009,9.729551,Books,0920668372,Love You Forever,Books Children's Books Classics,"cf,gateway,popular",0.392223,0.000000,0.736500,0.639616,0.342077,normal,AFZTV7WTRNPPLF2BSJ4C2VBQRMOQ
8,Books||0694003611,0.384753,0.000000,18.364449,0.000000,Books,0694003611,Goodnight Moon,Books Children's Books Animals,"cf,popular",0.470238,0.000000,0.749870,0.000000,0.332477,normal,AFZTV7WTRNPPLF2BSJ4C2VBQRMOQ
9,Books||B0066U1SJU,0.000000,2.411397,0.000000,0.000000,Books,B0066U1SJU,Chicka Chicka Boom Boom,Books Children's Books Literature & Fiction,semantic_user,0.000000,1.000000,0.000000,0.000000,0.330000,normal,AFZTV7WTRNPPLF2BSJ4C2VBQRMOQ


In [ ]:
# -------------------------
# Build small eval samples for normal, cold-start, and simulated cross-domain
# -------------------------

val_tmp = val_df.copy()
val_tmp["item_id"] = val_tmp.apply(lambda r: make_item_id(r["domain"], r["parent_asin"]), axis=1)
val_tmp["train_domains"] = val_tmp["user_id"].map(user_train_domains)

# Normal: target domain already appears in user train domains.
normal_eval_pool = val_tmp[val_tmp.apply(lambda r: r["domain"] in (r["train_domains"] or []), axis=1)].copy()

# Cold-start: use normal pool but hide user history at recommendation time.
cold_eval_pool = normal_eval_pool.copy()

# Simulated cross-domain: target domain has a different source domain available in train.
def choose_other_domain(row):
    domains = row["train_domains"] or []
    others = [d for d in domains if d != row["domain"]]
    return others[0] if others else None

cross_eval_pool = val_tmp.copy()
cross_eval_pool["source_domain"] = cross_eval_pool.apply(choose_other_domain, axis=1)
cross_eval_pool = cross_eval_pool[cross_eval_pool["source_domain"].notna()].copy()

print("normal pool:", normal_eval_pool.shape)
print("cold pool:", cold_eval_pool.shape)
print("cross-domain simulated pool:", cross_eval_pool.shape)

N_EVAL = 25  # keep small while iterating
normal_sample = normal_eval_pool.sample(min(N_EVAL, len(normal_eval_pool)), random_state=1).reset_index(drop=True)
cold_sample = cold_eval_pool.sample(min(N_EVAL, len(cold_eval_pool)), random_state=2).reset_index(drop=True)
cross_sample = cross_eval_pool.sample(min(N_EVAL, len(cross_eval_pool)), random_state=3).reset_index(drop=True)

normal_sample[["user_id", "domain", "parent_asin", "rating"]].head()

In [ ]:
# -------------------------
# Evaluation helpers
# -------------------------

def evaluate_candidate_function(sample: pd.DataFrame, route: str, top_n: int = TOP_CANDIDATES, k: int = TOP_K, use_onboarding: bool = True) -> pd.DataFrame:
    records = []
    for _, row in tqdm(sample.iterrows(), total=len(sample), desc=f"eval:{route}"):
        user_id = str(row["user_id"])
        target_domain = str(row["domain"])
        target_id = make_item_id(row["domain"], row["parent_asin"])

        try:
            if route == "normal":
                cands = normal_candidates(user_id, target_domain=target_domain, top_n=top_n)
            elif route == "cold_start":
                onboarding_text = make_simulated_onboarding_text(user_id) if use_onboarding else None
                cands = cold_start_candidates(
                    target_domain=target_domain,
                    user_id=user_id,
                    onboarding_text=onboarding_text,
                    use_onboarding=use_onboarding,
                    use_gateway=True,
                    top_n=top_n,
                )
            elif route == "cross_domain":
                source_domain = str(row["source_domain"])
                cands = cross_domain_candidates(user_id, source_domain=source_domain, target_domain=target_domain, top_n=top_n)
            else:
                raise ValueError(f"Unknown route: {route}")

            candidate_ids = cands["item_id"].tolist() if len(cands) else []
            m10 = hit_ndcg(candidate_ids, target_id, k=k)
            records.append({
                "user_id": user_id,
                "route": route,
                "target_domain": target_domain,
                "target_item_id": target_id,
                "n_candidates": len(candidate_ids),
                "recall_at_30": recall_at_k(candidate_ids, target_id, k=top_n),
                "hit_at_10": m10["hit"],
                "ndcg_at_10": m10["ndcg"],
                "target_rank": candidate_ids.index(target_id) + 1 if target_id in candidate_ids else None,
                "top_sources": cands["sources"].head(5).tolist() if len(cands) else [],
            })
        except Exception as e:
            records.append({
                "user_id": user_id,
                "route": route,
                "target_domain": target_domain,
                "target_item_id": target_id,
                "error": repr(e),
            })

    out = pd.DataFrame(records)
    valid = out[out.get("error").isna()] if "error" in out else out
    if len(valid):
        print(
            f"{route}: n={len(valid)} "
            f"Recall@{top_n}={valid['recall_at_30'].mean():.4f} "
            f"Hit@{k}={valid['hit_at_10'].mean():.4f} "
            f"NDCG@{k}={valid['ndcg_at_10'].mean():.4f}"
        )
    return out

# Run quick candidate-only checks. These do not call the LLM.
normal_results = evaluate_candidate_function(normal_sample, route="normal")
cold_results_no_onboarding = evaluate_candidate_function(cold_sample, route="cold_start", use_onboarding=False)
cold_results_with_onboarding = evaluate_candidate_function(cold_sample, route="cold_start", use_onboarding=True)
cross_results = evaluate_candidate_function(cross_sample, route="cross_domain")

In [ ]:
# Inspect errors or successful candidate hits.
all_candidate_results = pd.concat([
    normal_results,
    cold_results_no_onboarding.assign(route="cold_start_no_onboarding"),
    cold_results_with_onboarding.assign(route="cold_start_with_onboarding"),
    cross_results,
], ignore_index=True)

all_candidate_results.groupby("route")[["recall_at_30", "hit_at_10", "ndcg_at_10"]].mean()

In [ ]:
# -------------------------
# Provider-switchable LLM call
# -------------------------

from getpass import getpass

# Uncomment one or both when needed. Keys are not saved in the notebook unless you hardcode them.
# os.environ["GEMINI_API_KEY"] = getpass("Enter Gemini API key: ")
# os.environ["DEEPSEEK_API_KEY"] = getpass("Enter DeepSeek API key: ")
# os.environ["LLM_PROVIDER"] = "gemini"   # or "deepseek"


def call_llm(
    prompt: str,
    provider: str = LLM_PROVIDER,
    model: Optional[str] = None,
    temperature: float = TEMPERATURE,
    max_output_tokens: int = MAX_OUTPUT_TOKENS,
    json_mode: bool = True,
    timeout: int = 90,
    retries: int = LLM_RETRIES,
) -> Dict[str, Any]:
    """Provider-switchable JSON-first LLM call for Gemini or DeepSeek."""
    provider = provider.lower().strip()
    last_error = None

    for attempt in range(retries + 1):
        try:
            if provider == "gemini":
                api_key = os.getenv("GEMINI_API_KEY")
                if not api_key:
                    raise RuntimeError("Missing GEMINI_API_KEY environment variable.")
                model = model or GEMINI_MODEL
                url = f"https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent"
                payload = {
                    "contents": [{"role": "user", "parts": [{"text": prompt}]}],
                    "generationConfig": {
                        "temperature": temperature,
                        "maxOutputTokens": max_output_tokens,
                    },
                }
                if json_mode:
                    payload["generationConfig"]["responseMimeType"] = "application/json"
                resp = requests.post(url, params={"key": api_key}, json=payload, timeout=timeout)
                resp.raise_for_status()
                data = resp.json()
                raw_text = data["candidates"][0]["content"]["parts"][0]["text"]

            elif provider == "deepseek":
                api_key = os.getenv("DEEPSEEK_API_KEY")
                if not api_key:
                    raise RuntimeError("Missing DEEPSEEK_API_KEY environment variable.")
                model = model or DEEPSEEK_MODEL
                url = "https://api.deepseek.com/chat/completions"
                payload = {
                    "model": model,
                    "messages": [{"role": "user", "content": prompt}],
                    "temperature": temperature,
                    "max_tokens": max_output_tokens,
                }
                if json_mode:
                    payload["response_format"] = {"type": "json_object"}
                resp = requests.post(
                    url,
                    headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"},
                    json=payload,
                    timeout=timeout,
                )
                resp.raise_for_status()
                data = resp.json()
                raw_text = data["choices"][0]["message"]["content"]
            else:
                raise ValueError("provider must be 'gemini' or 'deepseek'")

            parsed = safe_json_loads(raw_text) if json_mode else None
            return {"raw_text": raw_text, "json": parsed, "provider": provider, "model": model}

        except Exception as e:
            last_error = e
            if attempt < retries:
                time.sleep(1.5 * (attempt + 1))
            else:
                raise last_error

In [ ]:
# -------------------------
# LLM reranker for top 30 -> top 10
# -------------------------

TASK_B_RERANK_PROMPT_VERSION = "task_b_rerank_v1_strict_json"


def build_task_b_user_context(user_id: str, source_domain: Optional[str] = None, max_examples: int = 8) -> Dict[str, Any]:
    hist = train_by_user.get(str(user_id))
    if hist is None or hist.empty:
        return {"user_id": str(user_id), "history_scope": "none", "summary": "No known train history."}
    if source_domain is not None:
        hist = hist[hist["domain"] == source_domain]
    if hist.empty:
        return {"user_id": str(user_id), "history_scope": f"source_domain={source_domain}", "summary": "No known history in source domain."}

    ratings = hist["rating"].astype(float)
    liked = hist[ratings >= 4.0].sort_values(["rating", "timestamp"], ascending=[False, False]).head(max_examples)
    disliked = hist[ratings <= 3.0].sort_values(["rating", "timestamp"], ascending=[True, False]).head(max(2, max_examples // 3))

    def ex(row):
        meta = item_lookup.get(row["item_id"], {})
        return {
            "domain": row["domain"],
            "title": clean_text(meta.get("title", ""), 140),
            "categories": clean_text(meta.get("categories", ""), 180),
            "rating": float(row["rating"]),
            "review": clean_text(row.get("review_text", ""), 300),
        }

    return {
        "user_id": str(user_id),
        "history_scope": "train_only" if source_domain is None else f"train_only_source_domain={source_domain}",
        "n_reviews": int(len(hist)),
        "domains_seen": sorted(hist["domain"].unique().tolist()),
        "mean_rating": float(ratings.mean()),
        "rating_distribution": {str(i): int((ratings.round().astype(int) == i).sum()) for i in range(1, 6)},
        "liked_examples": [ex(r) for _, r in liked.iterrows()],
        "lower_rated_examples": [ex(r) for _, r in disliked.iterrows()],
    }


def make_task_b_rerank_prompt(
    user_context: Dict[str, Any],
    candidates: pd.DataFrame,
    route: str,
    target_domain: Optional[str] = None,
    top_k: int = TOP_K,
    locale_hint: Optional[str] = None,
) -> str:
    cand_rows = []
    for i, row in candidates.head(TOP_CANDIDATES).reset_index(drop=True).iterrows():
        cand_rows.append({
            "candidate_number": int(i + 1),
            "domain": row["domain"],
            "parent_asin": row["parent_asin"],
            "title": clean_text(row.get("title", ""), 160),
            "categories": clean_text(row.get("categories", ""), 220),
            "retrieval_sources": row.get("sources", ""),
            "cf_score": float(row.get("cf_score", 0.0)),
            "semantic_score": float(row.get("semantic_score", 0.0)),
            "popularity_score": float(row.get("popularity_score", 0.0)),
            "gateway_score": float(row.get("gateway_score", 0.0)),
            "candidate_generator_rank": int(i + 1),
        })

    locale_instruction = ""
    if locale_hint:
        locale_instruction = f"Locale/context hint: {locale_hint}. Use it only if relevant. Avoid forced slang or stereotypes."

    return f"""
You are a recommendation reranker for an Amazon Books and Movies_and_TV recommender.

Task:
Choose the best {top_k} recommendations from the candidate list for this user.
Do not create new items. Only choose from the candidate list.

Route: {route}
Target domain: {target_domain or "not specified"}
{locale_instruction}

Ranking principles:
- Prefer items that fit the user's demonstrated tastes and rating behaviour.
- Use candidate generator scores as evidence, but you may reorder when the user context suggests it.
- For cold_start, prefer safe, broadly accessible, high-quality gateway items.
- For cross_domain, prefer items whose abstract themes/preferences transfer from the user's source-domain history.
- Keep some diversity. Do not choose near-duplicates unless strongly justified.

JSON rules:
- Return exactly one valid JSON object and nothing else.
- Do not use markdown fences.
- Do not include comments.
- Do not include trailing commas.
- Do not use NaN, Infinity, undefined, or null.

Return this exact JSON structure:
{{
  "recommendations": [
    {{
      "rank": 1,
      "candidate_number": 1,
      "domain": "Books or Movies_and_TV",
      "parent_asin": "...",
      "title": "...",
      "reason": "Short reason grounded in user context and candidate evidence."
    }}
  ],
  "rerank_notes": "Brief explanation of the ranking strategy."
}}

User context:
{json.dumps(user_context, ensure_ascii=False)}

Candidates:
{json.dumps(cand_rows, ensure_ascii=False)}
""".strip()


def rerank_candidates_with_llm(
    user_id: str,
    candidates: pd.DataFrame,
    route: str,
    target_domain: Optional[str] = None,
    source_domain: Optional[str] = None,
    provider: str = LLM_PROVIDER,
    model: Optional[str] = None,
    locale_hint: Optional[str] = None,
    top_k: int = TOP_K,
) -> Dict[str, Any]:
    user_context = build_task_b_user_context(user_id, source_domain=source_domain)
    prompt = make_task_b_rerank_prompt(
        user_context=user_context,
        candidates=candidates,
        route=route,
        target_domain=target_domain,
        top_k=top_k,
        locale_hint=locale_hint,
    )
    result = call_llm(prompt, provider=provider, model=model, json_mode=True, temperature=0.1, max_output_tokens=MAX_OUTPUT_TOKENS)
    payload = result["json"]

    # Convert LLM rankings to a dataframe for quick evaluation.
    recs = payload.get("recommendations", [])
    reranked = []
    candidate_by_key = {(str(r.domain), str(r.parent_asin)): r._asdict() for r in candidates.itertuples(index=False)}
    for rec in recs:
        key = (str(rec.get("domain", "")), str(rec.get("parent_asin", "")))
        base = candidate_by_key.get(key, {})
        reranked.append({
            **base,
            "llm_rank": int(rec.get("rank", len(reranked) + 1)),
            "llm_reason": clean_text(rec.get("reason", ""), 400),
        })

    reranked_df = pd.DataFrame(reranked).sort_values("llm_rank") if reranked else pd.DataFrame()
    return {
        "provider": result["provider"],
        "model": result["model"],
        "prompt_version": TASK_B_RERANK_PROMPT_VERSION,
        "raw": payload,
        "reranked_df": reranked_df,
    }

In [ ]:
# -------------------------
# Single LLM rerank test
# -------------------------
# Set one API key first, then uncomment.

# row = normal_sample.iloc[0]
# uid = row["user_id"]
# target_domain = row["domain"]
# target_id = make_item_id(row["domain"], row["parent_asin"])
# cands = normal_candidates(uid, target_domain=target_domain, top_n=TOP_CANDIDATES)
#
# reranked = rerank_candidates_with_llm(
#     user_id=uid,
#     candidates=cands,
#     route="normal",
#     target_domain=target_domain,
#     provider="gemini",  # or "deepseek"
#     locale_hint=None,
# )
# display(reranked["reranked_df"][["llm_rank", "domain", "parent_asin", "title", "sources", "llm_reason"]].head(10))
#
# llm_candidate_ids = reranked["reranked_df"]["item_id"].tolist()
# print("GROUND TRUTH:", target_id)
# print("LLM Hit@10 / NDCG@10:", hit_ndcg(llm_candidate_ids, target_id, k=10))

In [ ]:
# -------------------------
# Tiny LLM rerank eval loop
# -------------------------

def evaluate_llm_rerank_small(sample: pd.DataFrame, route: str = "normal", n: int = 5, provider: str = LLM_PROVIDER) -> pd.DataFrame:
    rows = sample.head(n).copy()
    records = []
    for _, row in tqdm(rows.iterrows(), total=len(rows), desc=f"llm-rerank:{route}"):
        user_id = str(row["user_id"])
        target_domain = str(row["domain"])
        target_id = make_item_id(row["domain"], row["parent_asin"])

        try:
            if route == "normal":
                cands = normal_candidates(user_id, target_domain=target_domain, top_n=TOP_CANDIDATES)
                source_domain = None
            elif route == "cross_domain":
                source_domain = str(row["source_domain"])
                cands = cross_domain_candidates(user_id, source_domain=source_domain, target_domain=target_domain, top_n=TOP_CANDIDATES)
            elif route == "cold_start":
                source_domain = None
                onboarding_text = make_simulated_onboarding_text(user_id)
                cands = cold_start_candidates(target_domain=target_domain, user_id=user_id, onboarding_text=onboarding_text, top_n=TOP_CANDIDATES)
            else:
                raise ValueError(route)

            before_ids = cands["item_id"].tolist()
            before = hit_ndcg(before_ids, target_id, k=TOP_K)

            reranked = rerank_candidates_with_llm(
                user_id=user_id,
                candidates=cands,
                route=route,
                target_domain=target_domain,
                source_domain=source_domain,
                provider=provider,
            )
            after_ids = reranked["reranked_df"]["item_id"].tolist() if len(reranked["reranked_df"]) else []
            after = hit_ndcg(after_ids, target_id, k=TOP_K)

            records.append({
                "user_id": user_id,
                "route": route,
                "target_item_id": target_id,
                "candidate_recall_at_30": recall_at_k(before_ids, target_id, TOP_CANDIDATES),
                "before_hit_at_10": before["hit"],
                "before_ndcg_at_10": before["ndcg"],
                "after_hit_at_10": after["hit"],
                "after_ndcg_at_10": after["ndcg"],
                "rerank_notes": reranked["raw"].get("rerank_notes", ""),
            })
        except Exception as e:
            records.append({"user_id": user_id, "route": route, "target_item_id": target_id, "error": repr(e)})
    return pd.DataFrame(records)

# Example after setting API key:
# llm_eval_df = evaluate_llm_rerank_small(normal_sample, route="normal", n=3, provider="gemini")
# llm_eval_df

## Fast iteration checklist

Recommended order:

1. Tune candidate generation first using `Recall@30`. If Recall@30 is weak, the LLM cannot recover the held-out item.
2. Compare normal, cold-start without onboarding, cold-start with simulated onboarding, and cross-domain.
3. Only then run the LLM reranker on 3–10 users to inspect relevance and whether NDCG@10 improves or drops.
4. Useful ablations:
   - CF only vs semantic only vs popularity only vs blended.
   - Cold-start: popularity only vs popularity + gateway vs popularity + gateway + onboarding semantic.
   - Cross-domain: popularity only vs semantic transfer query + popularity + gateway.
5. Once stable, convert the best candidate generators and reranker prompt into API-ready scripts.